# 02 - Create Virtual Stations

## Objective

Generate 600 virtual monitoring stations inside the Ganga Buffer.

These stations will later be used to extract:

- Sentinel/Landsat features
- VIIRS features
- LULC information

### Output

- Virtual_Stations.csv
- Virtual_Stations.shp


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from shapely.geometry import Point

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_PATH = PROJECT_ROOT
RIVER_PATH = DATA_ROOT / "boundaries" / "ganga_kanpur_varanasi.shp"
print(f"River boundary input: {RIVER_PATH}")


In [ ]:
river = gpd.read_file(RIVER_PATH)

print("="*60)
print("KANPUR → VARANASI RIVER")
print("="*60)

print(f"Features      : {len(river)}")
print(f"CRS           : {river.crs}")
print(f"Geometry Type : {river.geometry.iloc[0].geom_type}")

display(river.head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

river.plot(ax=ax, color="blue", linewidth=2)

ax.set_title("Kanpur → Varanasi River Reach")

plt.show()

In [ ]:
river_metric = river.to_crs(epsg=32644)

In [ ]:
total_length_m = river_metric.geometry.length.iloc[0]

total_length_km = total_length_m / 1000

print(f"River Length : {total_length_km:.2f} km")

In [ ]:
print(type(river_metric.geometry.iloc[0]))

In [ ]:
# ==========================================
# GENERATE MONITORING STATIONS
# ==========================================

from shapely.geometry import Point

STATION_SPACING = 1000  # meters

river_line = river_metric.geometry.iloc[0]

stations = []

distance = 0

while distance <= river_line.length:

    point = river_line.interpolate(distance)

    stations.append(point)

    distance += STATION_SPACING

print(f"Total Stations Generated: {len(stations)}")

In [ ]:
#Convert those points into a GeoDataFrame.

stations_gdf = gpd.GeoDataFrame(
    geometry=stations,
    crs=river_metric.crs
)

stations_gdf["Station_ID"] = [
    f"ST{i:04d}" for i in range(1, len(stations_gdf)+1)
]

stations_gdf = stations_gdf[["Station_ID", "geometry"]]

stations_gdf.head()

In [ ]:
#Convert back to latitude/longitude.

stations_gdf = stations_gdf.to_crs(epsg=4326)

In [ ]:
#extract coordinates
stations_gdf["Longitude"] = stations_gdf.geometry.x
stations_gdf["Latitude"] = stations_gdf.geometry.y

stations_gdf.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

river.to_crs(4326).plot(ax=ax, color="blue", linewidth=2)

stations_gdf.plot(
    ax=ax,
    color="red",
    markersize=8
)

plt.title("Monitoring Stations (1 km spacing)")
plt.show()

In [ ]:
output_folder = OUTPUT_ROOT / "monitoring_stations"

output_folder.mkdir(parents=True, exist_ok=True)

stations_gdf.to_file(
    output_folder / "Monitoring_Stations.shp"
)

stations_gdf.drop(columns="geometry").to_csv(
    output_folder / "Monitoring_Stations.csv",
    index=False
)

print("✅ Monitoring stations saved successfully.")